In [1]:
import torch

model1_pth = r'D:\HUST\2025.1\vhdl\model1_no_bin\model1.pth'
model1_state_dict = torch.load(model1_pth)

ModuleNotFoundError: No module named 'torch'

In [2]:
for name, tensor in model1_state_dict.items():
    if isinstance(tensor, torch.Tensor):
        print(f"{name} | shape={tuple(tensor.shape)}, dtype={tensor.dtype}")
    else:
        print(f"{name} | type={type(tensor)}")

fc1.weight | shape=(64, 784), dtype=torch.float32
fc1.bias | shape=(64,), dtype=torch.float32
fc2.weight | shape=(10, 64), dtype=torch.float32
fc2.bias | shape=(10,), dtype=torch.float32


In [3]:
from load_data import take_dataset

train_dataset, test_dataset = take_dataset(
        download=True
        # no binarization
    )

x0, y0 = train_dataset[0]
print(type(x0), type(y0))
print(y0)
print(x0.dtype)

<class 'torch.Tensor'> <class 'int'>
5
torch.float32


tensor([-0.0007, -0.0015,  0.0014, -0.0003,  0.0004,  0.0016, -0.0005,  0.0001,
        -0.0005, -0.0009, -0.0008, -0.0012, -0.0014, -0.0012, -0.0006,  0.0015,
         0.0008,  0.0014, -0.0010,  0.0011], device='cuda:0')
tensor([-11719, -24776,  23109,  -4301,   6798,  26253,  -7971,   1832,  -8129,
        -15760, -13605, -20830, -23491, -20559, -10254,  25459,  13048,  24127,
        -16897,  17952], device='cuda:0', dtype=torch.int32)
tensor([-0.0007, -0.0015,  0.0014, -0.0003,  0.0004,  0.0016, -0.0005,  0.0001,
        -0.0005, -0.0009, -0.0008, -0.0012, -0.0014, -0.0012, -0.0006,  0.0015,
         0.0008,  0.0014, -0.0010,  0.0011], device='cuda:0')


In [5]:
best, results = recommend_qmn_from_tensors(
    model1_state_dict,
    use_percentile=0.999,  # thử 0.999 trước (p99.9)
    radius=2,              # chỉ ~5 case
    clip_threshold=0.0,    # ưu tiên không clip
    verbose=True
)

m, n = best["m"], best["n"]
print("FINAL PICK:", (m, n))


[fc1.weight] shape=(64, 784), numel=50176
  min/max      : -3.54393 / 2.12222
  mean/std     : -0.0222559 / 0.398362
  abs_max      : 3.54393
  abs_p0.99    : 1.34368
  abs_p0.999    : 2.18219
  abs_p0.9999    : 2.77409

[fc1.bias] shape=(64,), numel=64
  min/max      : -0.830187 / 0.852854
  mean/std     : -0.0910002 / 0.305598
  abs_max      : 0.852854
  abs_p0.99    : 0.838574
  abs_p0.999    : 0.851426
  abs_p0.9999    : 0.852711

[fc2.weight] shape=(10, 64), numel=640
  min/max      : -2.62478 / 1.03377
  mean/std     : -0.227816 / 0.496171
  abs_max      : 2.62478
  abs_p0.99    : 1.51279
  abs_p0.999    : 2.25245
  abs_p0.9999    : 2.58755

[fc2.bias] shape=(10,), numel=10
  min/max      : -1.30044 / 2.1118
  mean/std     : -0.0854544 / 0.987989
  abs_max      : 2.1118
  abs_p0.99    : 2.03878
  abs_p0.999    : 2.1045
  abs_p0.9999    : 2.11107

[NARROW] use abs_p0.999 global=2.25245 -> anchor n0=29 -> candidates=[27, 28, 29, 30]

[SWEEP RESULTS] (sorted)
  Q3.29  rmse=3.23277e

In [6]:
with open("weights.hex", "w") as f:
    for x in w_q24_8.flatten():
        f.write(f'{int(x) & 0xFFFFFFFF:08x}\n')

In [5]:
export_state_dict_and_input_to_mem(
    state_dict=model1_state_dict,
    x=x0, # y0 = 5
    log_dir="weights_hex/model1_no_binaried",
    m=8,
    n=24,
)

Saved fc1.weight -> weights_hex\model1_no_binaried\fc1_weight.mem
Saved fc1.bias -> weights_hex\model1_no_binaried\fc1_bias.mem
Saved fc2.weight -> weights_hex\model1_no_binaried\fc2_weight.mem
Saved fc2.bias -> weights_hex\model1_no_binaried\fc2_bias.mem
Saved input -> weights_hex\model1_no_binaried\input.mem


In [7]:
from model import model1

model = model1()
state_dict = torch.load(model1_pth, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

with torch.no_grad():
    x_batch = x0.reshape(1, 1, 28, 28)
    logits = model(x_batch[:, 0])         # [1,10]
    print("logits FP32:", logits[0])
    logits_int = float_to_qm_n(logits, m=8, n=24)
    print("logits int:", logits_int[0])

logits FP32: tensor([-53.5463, -39.1347, -36.8562,   0.3126, -63.1379,  13.6029, -40.8609,
        -30.5381, -28.9001, -16.3094])
logits int: tensor([ -898357760,  -656571392,  -618343872,     5244521, -1059278528,
          228218304,  -685531328,  -512343904,  -484862688,  -273625760],
       dtype=torch.int32)


In [9]:
with open("output_logits_python.hex", "w") as f:
    for x in logits_int.flatten():
        f.write(f'{int(x) & 0xFFFFFFFF:08x}\n')